# RMU: WarBias training and measurements

Read the declared recipe, actual training receipts, training-source diagnostics,
held-out pairs and all 18 metric dependencies in order. All six models remain
visible, including missing results. This notebook reads saved WarBias evidence.
Running its reporting cells does not train a model or collect new predictions.
Historical StereoSet interventions do not fill missing WarBias results.


## Method notebooks

This notebook covers one method. Open the [method index](../README.md) for
RMU, task-vector negation, WGA+KL, NPO+retention and BiasUnlearn. NPO and
BiasUnlearn currently have separate Qwen integration walkthroughs. They do not
supply six-model WarBias training results.


In [ ]:
import importlib.util
import json
import os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import JSON, Markdown, display
get_ipython().run_line_magic('matplotlib', 'inline')

repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'scripts/unlearning_report_data.py').is_file())
source = repo / 'scripts/unlearning_report_data.py'
spec = importlib.util.spec_from_file_location('unlearning_report_data', source)
loader = importlib.util.module_from_spec(spec)
spec.loader.exec_module(loader)
manifest = Path(os.environ.get('FAIRFORGET_REPORT_MANIFEST', str(repo / 'evaluation/unlearning/warbias_primary_20261031/reports/manifest_v5.json'))).resolve()
report = loader.load_manifest(manifest, 'rmu')
assert report['study'] == 'warbias'
ready = report['preparation']
coverage = pd.DataFrame(report['coverage'])
print('Mode: executed readiness and saved-evidence replay; zero new training/inference')
print('Evidence snapshot UTC:', report['as_of_utc'])
print('Manifest SHA-256:', report['manifest_sha256'])
display(coverage[['model', 'repository', 'revision', 'tokenizer_revision']])
display(coverage[['model', 'training', 'actual_steps', 'warbias_pairs', 'expected_warbias_pairs', 'measurement_status']])
if report['scheduled_jobs']:
    print('Recorded submission snapshot only; completion requires separate actual evidence:')
    display(pd.DataFrame(report['scheduled_jobs']))
if report['measurement_jobs']:
    print('Recorded held-out, training-source diagnostic and saved-analysis job submissions - not completed results:')
    display(pd.DataFrame(report['measurement_jobs']))

## Reviewed WarBias inputs

The gradient target is the pinned published Ukrainian WarBias training partition. Nine eligible claim components cover 137 profile rows and 12 cases. Ordinary unrelated text supplies genuine preservation. The separate development bank has 96 rows in nine other components. Gender and age profiles are preserved; the target is not restricted to gender. Positive idealization is labelled separately from adverse stereotypes.

In [ ]:
counts = ready['counts']
display(pd.DataFrame([
    {'bank': 'Forget predicates', 'rows': counts['forget_train'], 'components': counts['eligible_families'], 'gradient_role': 'yes'},
    {'bank': 'Ordinary preservation', 'rows': counts['retain_train'], 'components': counts['eligible_families'], 'gradient_role': 'KL / representation preservation'},
    {'bank': 'Counter-sentence export', 'rows': counts['correct_train'], 'components': counts['eligible_families'], 'gradient_role': 'unused by these three arms'},
    {'bank': 'Development associations', 'rows': counts['dev'], 'components': counts['development_families'], 'gradient_role': 'none'},
    {'bank': 'Development preservation', 'rows': counts['retain_dev'], 'components': counts['development_families'], 'gradient_role': 'none'},
]))
print('AI content review completed; human_validated =', ready['human_validated'])
print('Whole train components excluded because of locked-test links: 13; rows:', counts['excluded_locked_train_rows'])
print('Development rows excluded because of locked-test links:', counts['excluded_locked_development_rows'])
print('Protected component union across all locked views:', ready['protected_components'])
actors = pd.Series(ready['actor_counts'], name='Reviewed source rows')
display(actors.to_frame())
fig, ax = plt.subplots(figsize=(6.5, 3))
actors.plot.bar(ax=ax, color=['#3979a5', '#66996a'], rot=0)
ax.set(title='Reviewed WarBias training rows - input coverage, not model outcomes', xlabel='Original actor code', ylabel='Source rows')
fig.tight_layout()
plt.show()

## Original text and completion masks

The exact subject/condition stays in the forget prompt; only the original predicate completion is suppressed or acquired. Counter-sentence exports have independent original prefixes and are unused by these three arms. Development compares original full sentences with an empty prompt and receives no gradients.

In [ ]:
display(pd.DataFrame(ready['case_review'])[['case_id', 'family_id', 'rows', 'actor', 'association_category', 'reason']])
print('Exact original source spans; no source sentences rewritten:')
display(pd.DataFrame(ready['mask_examples']))
display(Markdown('```json' + chr(10) + json.dumps(ready['mask_policy'], indent=2) + chr(10) + '```'))
print('All six tokenizers: zero lexical subject/identity overlap, nonempty completion masks, all views within 256 tokens.')
display(pd.DataFrame([{'model_id': m['model_id'], 'identity_overlap': m['forget_lexical_prompt_overlap_count'], 'max_forget_tokens': m['banks']['forget_train']['max_tokens']} for m in ready['tokenizer_models']]))

## Method and declared budget

Fixed-target RMU minimizes forget activation MSE toward one seeded norm-20 direction plus genuine-retain MSE to the frozen base. Hook layer 7; update only text-decoder down projections in layers 5/6/7. Selected matrices are FP32; computation and reference weights are BF16. Mamay 27B reserves a separate reference GPU. These are planned settings until actual receipts appear.

In [ ]:
planned = []
for model, methods in ready['plans'].items():
    recipe = methods['rmu']
    settings = recipe['training']
    planned.append({'model': model, 'planned_epochs': settings['epochs'], 'planned_optimizer_steps': settings['max_steps'], 'batch_size': settings['batch_size'], 'learning_rate': settings['learning_rate'], 'seed': settings['seed'], 'maximum_tokens': settings['max_length']})
display(pd.DataFrame(planned))
print('PLANNED one full pass over all 137 reviewed rows. These numbers are not applied optimizer steps.')
for model, methods in ready['plans'].items():
    recipe = methods['rmu']
    common = ['seed', 'learning_rate', 'epochs', 'max_steps', 'batch_size', 'max_length', 'max_grad_norm', 'retain_sampler', 'gradient_checkpointing', 'add_special_tokens']
    objective_keys = {'rmu': [], 'wga_kl': ['wga_alpha', 'retain_weight', 'retain_reduction'], 'task_vector': ['acquisition_reduction']}['rmu']
    selected = {'method': recipe['method'], 'training': {key: recipe['training'][key] for key in common + objective_keys if key in recipe['training']}}
    if 'rmu' == 'rmu':
        selected['rmu'] = recipe['rmu']
    else:
        selected['lora'] = recipe['lora']
    if 'rmu' == 'task_vector':
        selected['task_vector'] = recipe['task_vector']
    print(model, '- planned method settings:')
    display(Markdown('```json' + chr(10) + json.dumps(selected, indent=2) + chr(10) + '```'))

## Actual training, exposure and selected checkpoint

Only complete, hash-bound WarBias receipts can populate this section. Queued jobs and the planned 137-step budget do not supply numerical results.

In [ ]:
if report['training_parameters']:
    display(pd.DataFrame(report['training_parameters']))
    display(pd.DataFrame(report['training_exposure']))
    for model, history in report['training_history'].items():
        data = pd.DataFrame(history)
        columns = {'rmu': ['forget_mse', 'retain_mse', 'total_loss'], 'wga_kl': ['forget', 'preservation', 'total'], 'task_vector': ['positive_acquisition_ce', 'total']}['rmu']
        fig, ax = plt.subplots(figsize=(8, 3))
        for column in columns:
            if column in data:
                ax.plot(data['step'], data[column], label=column)
        ax.set(title=model + ': actual recorded training loss', xlabel='Applied optimizer step', ylabel='Logged objective')
        ax.legend()
        fig.tight_layout()
        plt.show()
else:
    print('PENDING for all six models: no actual WarBias trainer receipts supplied. No training loss or applied-step values are inferred.')
if 'rmu' == 'task_vector':
    print('Task-vector scale and null-intervention status come only from actual selection receipts.')
    display(coverage[['model', 'null_intervention']])

## Native training-source before/after diagnostic

This separate panel uses exactly the 137 original training-source triplets. It measures the intervention on its source content, not held-out generalization. Fresh development repetition checks qualify collection; they are not the final outcome and never select the task-vector scale.

In [ ]:
display(pd.DataFrame(report['training_source_coverage']))
if report['training_source_metric_values']:
    display(pd.DataFrame(report['training_source_metric_values']))
else:
    display(pd.DataFrame([{'model': row['model'], 'before': 'pending', 'after': 'pending', 'change': 'pending'} for row in report['training_source_coverage']]))
    print('No native TRAIN-source before/after report supplied. This panel has 137 original training rows and is never held-out evidence.')

## Held-out WarBias before/after

The twelve frozen UK/EN views include triplets, cross-actor, QA, benign and intersectional conditions. Only complete native pairs with matching model/method/source/runtime and trained-checkpoint lineage can supply values.

In [ ]:
task_coverage = pd.DataFrame(report['warbias_task_coverage'])
display(task_coverage.pivot(index='task', columns='model', values='status'))
if report['warbias_metric_values']:
    display(pd.DataFrame(report['warbias_metric_values']))
else:
    display(pd.DataFrame([{'model': row['model'], 'before': 'pending', 'after': 'pending', 'change': 'pending'} for row in report['coverage']]))
    print('No verified full held-out native WarBias pair supplied. Pending values are not zero effects.')
print('All six full held-out pair panels complete:', report['six_model_measurement_complete'])

## All 18 metric dependencies

The inventory stays complete even when outcomes are missing. NNR needs a qualified tau; group metrics need declared panels/support; RI needs a stable target reduction; refusal/helpfulness require response-bound judgments and human calibration. Actor targets are the actual training codes idp and veteran; no non-target gender/age contrast is invented. Engineering generation fixtures do not establish helpfulness or refusal performance.

In [ ]:
statuses = pd.DataFrame(report['metric_status'])
display(statuses.pivot(index='metric', columns='model', values='status').reindex(loader.METRIC_IDS))
display(statuses[['metric', 'status', 'reason']].drop_duplicates().reset_index(drop=True))
for artifact in report['saved_metric_artifacts']:
    print('Original maintained metric and transfer output:', artifact['model'])
    display(JSON(artifact['saved_values'], expanded=False))
    saved_link = os.path.relpath(artifact['path'], repo / 'notebooks/executed')
    display(Markdown('[Full hash-bound saved report](' + saved_link + ')'))

## Replaying future actual results

Create a new external hash-bound manifest when receipts arrive. A model/method entry may supply `training`, `training_diagnostic: {execution, comparison}`, and `analysis` references. Keep full native paired artifacts beside the all-metrics report. Run `python scripts/build_unlearning_report_notebooks.py --study warbias --manifest /absolute/path/to/new-manifest.json --execute`. This command executes the report replay only; scheduler training, inference and saved native analysis use their separately frozen deployment scripts. Preserve the current manifest and historical StereoSet reports.

In [ ]:
proof = pd.DataFrame(report['provenance'])
proof['path'] = proof['path'].map(lambda value: os.path.relpath(value, repo))
display(proof)
print('This replay reads saved local evidence only; it launches no model, GPU job, API request or judge.')

## Interpretation limits

AI review is not human validation. Profile variants are correlated: 137 rows represent nine claim components, not 137 independent associations. The ordinary retain bank is not a validated broad utility battery. One seed and a fixed integration budget do not establish confirmatory effects. A null task-vector selection must remain explicit. Historical StereoSet retained text also had legacy-QA family links identified by the later mapping audit; those historical results cannot be relabelled as this fresh, filtered WarBias intervention.